# Multimodal Prediction with AutoGluon

Multimodal prediction combines **text, images, and tabular features** in a single model. AutoGluon automatically detects what type of data is in each column and routes it through the appropriate encoder (transformer for text/images, MLP for tabular), then fuses the representations.

Why multimodal? Adding modalities almost always improves accuracy when they provide complementary signals. For example, a product's photo and its textual description together predict quality better than either alone.

This notebook covers:
1. How AutoGluon detects modalities from column values
2. Training on the PetFinder dataset (tabular + images)
3. Modality ablation: comparing unimodal vs. multimodal performance
4. Handling missing values in mixed-modality data
5. Practical gotchas

In [ ]:
import autogluon
print('AutoGluon version:', autogluon.__version__)

## 1. How AutoGluon Detects Modalities

AutoGluon inspects the **values** in each column to decide the data type:

| Column content | Detected as |
|---|---|
| Numbers (int/float) | Tabular — numerical |
| Short strings with few unique values | Tabular — categorical |
| Variable-length strings (>10 chars, many unique) | **Text** |
| Strings ending in `.jpg`, `.png`, `.tiff`, etc. that are valid paths | **Image** |
| Strings ending in `.pdf`, `.docx` | **Document** |

You can override the detection with `column_types={'col': 'text'}` or `{'col': 'image'}`.

In [ ]:
import os, zipfile, urllib.request
import pandas as pd

DATA_URL = 'https://autogluon.s3.amazonaws.com/datasets/petfinder_and_ner.zip'
DATA_DIR = './petfinder'

if not os.path.exists(DATA_DIR):
    print('Downloading PetFinder dataset...')
    urllib.request.urlretrieve(DATA_URL, './petfinder.zip')
    with zipfile.ZipFile('./petfinder.zip', 'r') as z:
        z.extractall('.')
    os.remove('./petfinder.zip')
    print('Done.')

train_df = pd.read_csv(os.path.join(DATA_DIR, 'petfinder_adoption_train.csv'), index_col=0)
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'petfinder_adoption_test.csv'), index_col=0)

print(f'Train: {train_df.shape} | Test: {test_df.shape}')
train_df.head(3)

In [ ]:
# Inspect the columns and their data types
print('Column dtypes:')
print(train_df.dtypes)
print('\nLabel distribution:')
print(train_df['AdoptionSpeed'].value_counts().sort_index())

In [ ]:
# The 'Images' column contains local file paths — make them absolute
img_base = os.path.abspath(DATA_DIR)

def fix_image_paths(df: pd.DataFrame, img_col: str = 'Images') -> pd.DataFrame:
    df = df.copy()
    df[img_col] = df[img_col].apply(
        lambda p: os.path.join(img_base, p) if pd.notna(p) and not os.path.isabs(p) else p
    )
    return df

train_df = fix_image_paths(train_df)
test_df  = fix_image_paths(test_df)

# Check how many images exist
n_valid = train_df['Images'].apply(lambda p: pd.notna(p) and os.path.exists(p)).sum()
print(f'{n_valid}/{len(train_df)} training images found on disk')

## 2. Handling Missing Values in Mixed-Modality Data

> **Critical:** NaN in an image or text column causes an error during training. You must handle missing values before fitting.

- **Missing image path** → drop the row, or replace with a blank/placeholder image
- **Missing text** → replace with an empty string `''`
- **Missing numeric value** → AutoGluon handles this automatically (median imputation)

In [ ]:
# Check for missing values
print('Missing values per column:')
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

In [ ]:
# Handle missing text (fill with empty string) and missing images (drop row)
text_cols = ['Description']
img_col   = 'Images'

for col in text_cols:
    if col in train_df.columns:
        train_df[col] = train_df[col].fillna('')
        test_df[col]  = test_df[col].fillna('')

# Drop rows where image file doesn't exist
train_df = train_df[train_df[img_col].apply(lambda p: pd.notna(p) and os.path.exists(p))].copy()
test_df  = test_df[test_df[img_col].apply(lambda p: pd.notna(p) and os.path.exists(p))].copy()

print(f'After cleaning — Train: {len(train_df)} | Test: {len(test_df)}')

## 3. Train with All Modalities

In [ ]:
from autogluon.multimodal import MultiModalPredictor

label = 'AdoptionSpeed'

multimodal_predictor = MultiModalPredictor(
    label=label,
    problem_type='multiclass',
    path='./ag_multimodal_model',
)

multimodal_predictor.fit(
    train_data=train_df,
    time_limit=300,          # give it more time — fusing modalities is expensive
)

In [ ]:
multimodal_metrics = multimodal_predictor.evaluate(test_df)
print('Multimodal metrics:', multimodal_metrics)

## 4. Modality Ablation — Does Each Modality Help?

A simple way to understand each modality's contribution: train with only a subset of columns and compare.

In [ ]:
# Tabular-only: drop text and image columns
tabular_cols = [c for c in train_df.columns if c not in ['Description', 'Images', label]]
tabular_cols.append(label)

tabular_predictor = MultiModalPredictor(
    label=label, problem_type='multiclass', path='./ag_tabular_only'
)
tabular_predictor.fit(train_df[tabular_cols], time_limit=120)
tabular_metrics = tabular_predictor.evaluate(test_df[tabular_cols])
print('Tabular-only metrics:', tabular_metrics)

In [ ]:
# Compare results
import pandas as pd
comparison = pd.DataFrame({
    'Setup': ['Tabular only', 'Tabular + Text + Image'],
    'Accuracy': [
        tabular_metrics.get('accuracy', float('nan')),
        multimodal_metrics.get('accuracy', float('nan')),
    ],
})
print(comparison.to_string(index=False))

## 5. Forcing Column Types

Sometimes AutoGluon misdetects a column. For example, a numeric ID column should not be treated as a meaningful feature, or a column with long text strings that happen to look like paths. Use `column_types` to override.

In [ ]:
# Example: force 'Description' to be treated as text even if it's short
# and force 'Images' to be treated as image paths
explicit_predictor = MultiModalPredictor(
    label=label,
    problem_type='multiclass',
    path='./ag_multimodal_explicit',
)
explicit_predictor.fit(
    train_data=train_df,
    time_limit=120,
    column_types={
        'Description': 'text',
        'Images': 'image',
    },
)
print('Explicit column types metrics:', explicit_predictor.evaluate(test_df))

## ⚠️ Practical Gotchas

| Gotcha | What Happens | Fix |
|--------|-------------|-----|
| **NaN in image or text column** | `TypeError` or `NullPointerException` mid-training | Fill text NaNs with `''`; drop or replace rows with missing image paths |
| **Image paths are relative** | Paths work at fit time but fail if working directory changes | Always use `os.path.abspath()` |
| **Short text treated as categorical** | AutoGluon assigns the wrong encoder | Override with `column_types={'col': 'text'}` |
| **A numeric ID column included as a feature** | Model overfits to arbitrary IDs | Drop ID-like columns before fitting |
| **Training time scales with number of modalities** | Multimodal training is significantly slower than tabular-only | Budget at least 5 min per modality; use `presets='medium_quality'` for faster runs |
| **GPU OOM with large images and transformers** | CUDA out of memory error | Reduce `hyperparameters={'env.per_gpu_batch_size': 4}` or resize images |
| **Modality imbalance** | One modality dominates and others are ignored | Check per-modality feature importance; consider separate encoders or weighting |